# 小时级交通预测 · Temporal Fusion Transformer (TFT)

本 notebook 是 [`model.ipynb`](model.ipynb)（CatBoost 主力）的 **时序探索对照实验**，对齐 [`model.md`](model.md) §7 与 [`SOLUTION.md`](../doc/SOLUTION.md) §6.3：用 `pytorch-forecasting` 的 **TemporalFusionTransformer** 在 `data_autobahn/` 上预测 **小时级总流量 `kfz_h`**（P10/P50/P90 分位）。

**与 CatBoost 的分工**

| | CatBoost（主力） | TFT（本 notebook，探索层） |
|---|---|---|
| 角色 | 4 年全程主交付 | 时序 SOTA 对照亮点 |
| 输入 | 表格特征 + 历史画像 | 编码器序列 + 已知未来协变量 |
| 远期推理 | ✅ 直接 lookup | ⚠️ 需 observed 序列（远期递归，误差累积） |
| 可解释 | SHAP | Variable Selection + Temporal Attention |

**TFT 三类输入**（对齐 `model.md` §7.1）
- **Static**：`site_id`, `road`, `direction`, `site_name`, `bab_km`, `longitude`, `latitude`
- **Time-varying KNOWN（未来可知）**：日历 sin/cos、`tagestyp`、`season`、假期标志/计数、天气气候态
- **Time-varying OBSERVED（仅历史）**：`kfz_h`（目标）, `sv_h`, `v_kfz`, `lt_mean`, `fbt_mean`

---

### 运行顺序
0. **环境准备**（venv + 安装 torch / lightning / pytorch-forecasting + 注册 kernel）
1. 库导入
2. **超参数配置**（§1.1 — 全部在此修改）
3. 路径与数据文件
4. 数据加载与清洗（与 `model.ipynb` 一致）
5. 特征工程（日历 + conditional）
6. 构建 `TimeSeriesDataSet` + DataLoader
7. TFT 训练 + Loss 曲线
8. 验证集评估与可视化
9. 可解释性（变量重要性 / 注意力）
10. 模型保存

> ⚠️ TFT 仅作历史回测对照，**不进 2026–2029 主交付关键路径**（远期 observed 输入不可得）。

## 0. 环境准备（venv）

首次运行先执行本 cell：复用仓库根目录 `.venv`，额外安装 TFT 依赖（`torch` / `lightning` / `pytorch-forecasting`），并注册 Jupyter kernel。

> 与 `model.ipynb` 共用同一个 `.venv`。执行完后请把右上角 kernel 切到 **`Python (autobahn .venv)`**，Restart Kernel，再从下一节继续。

In [ ]:
import subprocess
import sys
import os
from pathlib import Path


# --- 定位仓库根目录（含 data_autobahn）---
def _find_repo_root() -> Path:
    starts = []
    nb = globals().get("__vsc_ipynb_file__")          # VS Code 注入的 notebook 路径
    if nb:
        starts.append(Path(nb).resolve().parent)
    try:
        starts.append(Path(os.getcwd()))
    except (PermissionError, OSError):
        pass
    starts.append(Path(__file__).resolve().parent if "__file__" in globals() else Path.home())
    for start in starts:
        for cand in [start, *start.parents]:
            if (cand / "data_autobahn").exists():
                return cand
    return starts[0] if starts else Path.home()


ROOT = _find_repo_root()
VENV_DIR = ROOT / ".venv"
KERNEL_NAME = "autobahn-venv"
KERNEL_DISPLAY = "Python (autobahn .venv)"

# TFT 额外依赖（在 model.ipynb 的 requirements.txt 之上补装）
TFT_REQUIREMENTS = [
    "torch>=2.1",
    "lightning>=2.2",
    "pytorch-forecasting>=1.0",
    "pandas>=2.0",
    "numpy",
    "matplotlib",
    "tqdm",
    "pyarrow",
    "ipykernel",
]


def _venv_python() -> Path:
    sub = "Scripts" if sys.platform == "win32" else "bin"
    return VENV_DIR / sub / ("python.exe" if sys.platform == "win32" else "python")


def _in_project_venv() -> bool:
    try:
        return VENV_DIR.resolve() in Path(sys.executable).resolve().parents
    except Exception:
        return False


def _run(cmd: list[str], **kwargs) -> None:
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True, **kwargs)


# 1) 创建/复用 venv
py = _venv_python()
if py.exists():
    print(f"✔ 虚拟环境已存在: {VENV_DIR}")
else:
    if VENV_DIR.exists():
        import shutil
        print(f"⚠ 检测到不完整的 .venv（缺少 {py.name}），删除后重建 ...")
        shutil.rmtree(VENV_DIR, ignore_errors=True)
    print(f"创建虚拟环境: {VENV_DIR}")
    _run([sys.executable, "-m", "venv", "--copies", str(VENV_DIR)])
    py = _venv_python()

if not py.exists():
    raise FileNotFoundError(f"venv python 未找到: {py}")

# 2) 安装依赖
print("\n安装/更新 TFT 依赖（torch / lightning / pytorch-forecasting，首次较慢）...")
_run([str(py), "-m", "pip", "install", "--upgrade", "pip"])
_run([str(py), "-m", "pip", "install", *TFT_REQUIREMENTS])

# 3) 注册 Jupyter kernel
print("\n注册 Jupyter kernel ...")
_run([
    str(py), "-m", "ipykernel", "install", "--user",
    "--name", KERNEL_NAME, "--display-name", KERNEL_DISPLAY,
])

# 4) 验证关键包
print("\n验证依赖版本:")
_run([str(py), "-c",
      "import torch, lightning, pytorch_forecasting as pf; "
      "print('torch              ', torch.__version__); "
      "print('lightning          ', lightning.__version__); "
      "print('pytorch_forecasting', pf.__version__); "
      "print('mps available      ', getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()); "
      "print('cuda available     ', torch.cuda.is_available()); "
      "print('deps OK')"])

print("-" * 56)
print(f"仓库根目录 : {ROOT}")
print(f"venv Python: {py}")
print(f"当前 kernel: {sys.executable}")
if _in_project_venv():
    print("✔ 当前已在项目 .venv 中，可直接运行后续 cell")
else:
    print("⚠ 当前 kernel 不在 .venv 中")
    print(f"  → 请切换 kernel 为: {KERNEL_DISPLAY}")
    print("  → Restart Kernel 后，从「库导入」cell 继续")

## 1. 库导入

> 依赖安装见 **§0 环境准备**。请确认 kernel 为 `Python (autobahn .venv)`。
> **超参数全部在下一节 §1.1 配置块中修改。**

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer, NaNLabelEncoder
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

print("torch", torch.__version__, "| lightning", pl.__version__)

## 1.1 超参数配置（★ 全部在此修改）

序列长度、TFT 网络、训练、评估、可视化的可调参数集中于此。

In [ ]:
# =============================================================================
# 超参数配置块 — 修改此处即可，无需在 notebook 其他位置找 magic number
# =============================================================================

# --- 随机种子 ---
RANDOM_SEED = 42

# --- 预测目标 ---
TARGET = "kfz_h"                       # TFT 聚焦主流量；sv_h/v_kfz 作为 observed 输入

# --- 时序切分（禁止随机划分；与 model.ipynb 对齐）---
TRAIN_END = pd.Timestamp("2024-12-31 23:59:59")   # 训练: 2023 ~ 2024
VAL_START = pd.Timestamp("2025-01-01 00:00:00")    # 验证: 2025 全年

# --- 序列窗口（小时）---
MAX_ENCODER_LENGTH = 7 * 24            # 编码器回看 7 天
MAX_PREDICTION_LENGTH = 24             # 解码器预测 1 天（多步）

# --- 分位数（QuantileLoss 输出 P10/P50/P90）---
QUANTILES = [0.1, 0.5, 0.9]
Q_IDX = {"p10": 0, "p50": 1, "p90": 2}

# --- TFT 网络结构 ---
TFT_PARAMS = dict(
    hidden_size=32,                    # 主隐藏维度（16/32/64）
    lstm_layers=1,
    attention_head_size=4,
    dropout=0.15,
    hidden_continuous_size=16,         # 连续变量嵌入维度
    learning_rate=3e-3,
)

# --- 训练 ---
BATCH_SIZE = 256
MAX_EPOCHS = 12                        # CPU/MPS 上控制时长；GPU 可调大
GRADIENT_CLIP_VAL = 0.1
EARLY_STOP_PATIENCE = 4
REDUCE_ON_PLATEAU_PATIENCE = 2
NUM_WORKERS = 0                        # macOS notebook 下置 0 更稳
LIMIT_TRAIN_BATCHES = 1.0             # 调试可设 0.2 仅跑部分批次加速
ACCELERATOR = "auto"                  # auto: 自动选 cuda/mps/cpu

# --- 预测后处理截断 ---
KFZ_CLIP_MIN = 0.0

# --- 评估指标 ---
MAPE_EPS = 1.0                         # MAPE 分母保护
PEAK_QUANTILE = 0.90                   # 峰值小时 top 10%
PICP_TARGET_PCT = 80.0                 # P10–P90 目标覆盖率 (%)

# --- 可视化 ---
PLOT_FIGSIZE = (13, 4.5)
LOSS_FIGSIZE = (6.5, 4.5)

# =============================================================================
pl.seed_everything(RANDOM_SEED, workers=True)

print("超参数已加载 ✔")
print(f"  目标         : {TARGET}")
print(f"  训练截止     : {TRAIN_END.date()}  |  验证起始: {VAL_START.date()}")
print(f"  encoder/decoder: {MAX_ENCODER_LENGTH}h / {MAX_PREDICTION_LENGTH}h")
print(f"  TFT          : hidden={TFT_PARAMS['hidden_size']}, heads={TFT_PARAMS['attention_head_size']}, dropout={TFT_PARAMS['dropout']}")
print(f"  分位数       : {QUANTILES}")

## 1.2 路径与数据文件

In [ ]:
import os


def find_root() -> Path:
    starts = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        starts.append(Path(nb).resolve().parent)
    try:
        starts.append(Path(os.getcwd()))
    except (PermissionError, OSError):
        pass
    starts.append(Path(__file__).resolve().parent if "__file__" in globals() else Path.home())
    for start in starts:
        for cand in [start, *start.parents]:
            if (cand / "data_autobahn").exists():
                return cand
    raise FileNotFoundError("未找到 data_autobahn 目录")


ROOT = find_root()
DATA_DIR = ROOT / "data_autobahn"
MODEL_DIR = ROOT / "models"
TFT_DIR = MODEL_DIR / "tft"            # TFT checkpoint / 导出
PROC_DIR = ROOT / "processed"
LOG_DIR = TFT_DIR / "lightning_logs"
for d in (MODEL_DIR, TFT_DIR, PROC_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

FILES = {
    "traffic": DATA_DIR / "合并表格，小时交通流量.csv",
    "temp": DATA_DIR / "合并表格，时间，气温，路温.csv",
    "holiday": DATA_DIR / "合并表格，holiday日级.csv",
    "weather": DATA_DIR / "合并表格，weather日级.csv",
    "construction": DATA_DIR / "合并表格，construction日级.csv",
    "events": DATA_DIR / "合并表格，special_events日级.csv",
}

print("ROOT     :", ROOT)
print("DATA_DIR :", DATA_DIR)
print("TFT_DIR  :", TFT_DIR)
for k, v in FILES.items():
    print(f"  {k:13s}: {'OK' if v.exists() else 'MISSING'}  {v.name}")

## 2. 数据加载与清洗

与 [`model.ipynb`](model.ipynb) §2 完全一致：所有表分隔符 `;`，**第 2 行是中文说明**（读取时跳过），部分列用逗号小数。

In [ ]:
def read_semicolon(path: Path) -> pd.DataFrame:
    """读取分号分隔表，跳过第 2 行中文说明。"""
    return pd.read_csv(path, sep=";", skiprows=[1], dtype=str, keep_default_na=True)


def to_num(series: pd.Series) -> pd.Series:
    """逗号小数 -> 浮点。"""
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False).replace({"nan": np.nan, "": np.nan}),
        errors="coerce",
    )


# ---------- 2.1 主表：小时交通流量 ----------
traffic = read_semicolon(FILES["traffic"])
for c in ["bab_km", "longitude", "latitude", "kfz_h", "sv_h", "v_kfz"]:
    traffic[c] = to_num(traffic[c])

traffic["ts"] = pd.to_datetime(
    traffic["datum"] + " " + traffic["t_start"], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)
traffic = traffic.dropna(subset=["ts"]).copy()
traffic["date"] = traffic["ts"].dt.normalize()
traffic["hour"] = traffic["ts"].dt.hour
traffic["weekday"] = traffic["wochentag"].astype(int)          # 1-7
traffic["site_id"] = traffic["road"] + "_" + traffic["direction"] + "_" + traffic["site_name"]

# 异常处理：物理不可能值 -> NaN
traffic.loc[traffic["kfz_h"] < 0, "kfz_h"] = np.nan
traffic.loc[traffic["sv_h"] < 0, "sv_h"] = np.nan
traffic.loc[(traffic["kfz_h"].isna()) | (traffic["kfz_h"] <= 0), "v_kfz"] = np.nan

print("主表行数:", len(traffic))
print("站点数  :", traffic["site_id"].nunique())
print("时间范围:", traffic["ts"].min(), "~", traffic["ts"].max())
traffic[["site_id", "ts", "hour", "weekday", "tagestyp", "kfz_h", "sv_h", "v_kfz"]].head()

In [ ]:
# ---------- 2.2 温度表（分钟）-> 小时聚合 ----------
temp_raw = read_semicolon(FILES["temp"])
temp_raw["lt"] = to_num(temp_raw["lt"])
temp_raw["fbt"] = to_num(temp_raw["fbt"])
temp_raw["ts"] = pd.to_datetime(temp_raw["t_start"], errors="coerce")
temp_raw = temp_raw.dropna(subset=["ts"]).copy()
temp_raw["date"] = temp_raw["ts"].dt.normalize()
temp_raw["hour"] = temp_raw["ts"].dt.hour

temp_hourly = (
    temp_raw.groupby(["date", "hour"])
    .agg(lt_mean=("lt", "mean"), fbt_mean=("fbt", "mean"), fbt_min=("fbt", "min"))
    .reset_index()
)
temp_hourly["month"] = temp_hourly["date"].dt.month
temp_climo = (
    temp_hourly.groupby(["month", "hour"])
    .agg(lt_mean_c=("lt_mean", "mean"), fbt_mean_c=("fbt_mean", "mean"), fbt_min_c=("fbt_min", "mean"))
    .reset_index()
)
print("温度小时表:", temp_hourly.shape, "| 气候态:", temp_climo.shape)
temp_hourly.head()

In [ ]:
# ---------- 2.3 conditional 日级表 ----------
def load_daily(path: Path, num_cols, cat_cols):
    df = read_semicolon(path)
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    for c in num_cols:
        if c in df.columns:
            df[c] = to_num(df[c]).fillna(0)
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].fillna("").astype(str)
    keep = ["date"] + [c for c in (num_cols + cat_cols) if c in df.columns]
    return df[keep].drop_duplicates("date")


holiday = load_daily(
    FILES["holiday"],
    num_cols=[
        "is_school_holiday_DE_BY", "is_school_holiday_AT_SB", "is_school_holiday_AT_TI",
        "is_public_holiday_DE_BY", "is_public_holiday_AT_SB", "is_public_holiday_AT_TI",
        "school_holiday_count", "public_holiday_count",
        "is_holiday_start", "is_holiday_end", "in_traffic_window",
    ],
    cat_cols=["window_direction", "window_risk_level", "a8_direction", "a93_direction"],
)

weather = read_semicolon(FILES["weather"])
weather["date"] = pd.to_datetime(weather["date"], errors="coerce").dt.normalize()
for c in ["precip_mm", "snowfall_mm", "low_vis_hours", "t_min_c", "t_max_c", "has_ice_risk",
          "precip_mm_mean", "low_vis_hours_mean", "ice_risk_prob", "t_min_c_mean", "t_max_c_mean"]:
    if c in weather.columns:
        weather[c] = to_num(weather[c])
weather["w_precip"] = weather["precip_mm"].fillna(weather.get("precip_mm_mean"))
weather["w_snow"] = weather["snowfall_mm"].fillna(0)
weather["w_lowvis"] = weather["low_vis_hours"].fillna(weather.get("low_vis_hours_mean"))
weather["w_tmin"] = weather["t_min_c"].fillna(weather.get("t_min_c_mean"))
weather["w_tmax"] = weather["t_max_c"].fillna(weather.get("t_max_c_mean"))
weather["w_ice"] = weather["has_ice_risk"].fillna(weather.get("ice_risk_prob"))
weather["weather_source"] = weather["weather_source"].fillna("climatology").astype(str)
weather = weather[["date", "w_precip", "w_snow", "w_lowvis", "w_tmin", "w_tmax", "w_ice", "weather_source"]]

construction = load_daily(
    FILES["construction"],
    num_cols=[
        "has_a8_construction", "has_a93_construction", "a8_construction_count", "a93_construction_count",
        "has_2_plus_0", "two_plus_0_count", "max_closed_lanes", "sum_closed_lanes",
        "has_target_bbox_construction",
    ],
    cat_cols=[],
)

events = load_daily(
    FILES["events"],
    num_cols=[
        "has_special_event", "active_event_count", "max_impact_level", "impact_score",
        "affects_a8_ost", "affects_a93_sued",
        "has_munich_event", "has_salzburg_event", "has_rosenheim_event", "has_kufstein_event",
        "has_confirmed_event", "has_estimated_event",
    ],
    cat_cols=[],
)

print("holiday     :", holiday.shape)
print("weather     :", weather.shape)
print("construction:", construction.shape)
print("events      :", events.shape)
holiday.head(3)

## 3. 特征工程

- **日历特征**：周期 sin/cos 编码（与 `model.ipynb` 一致）
- **conditional 合并**：假期/天气/温度/施工/事件按 `date` left join
- **连续小时面板**：按 `site_id` 把时间补成无间断的小时序列 → 生成连续 `time_idx`（TFT 必需），缺测目标用「站点×小时×星期」中位数回填并打 `is_imputed` 标记

In [ ]:
# ---------- 3.1 日历特征 ----------
def add_calendar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    d = df["date"]
    df["month"] = d.dt.month
    df["doy"] = d.dt.dayofyear
    df["week_of_year"] = d.dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["weekday"] >= 6).astype(int)
    df["is_friday"] = (df["weekday"] == 5).astype(int)
    df["is_saturday"] = (df["weekday"] == 6).astype(int)
    df["is_sunday"] = (df["weekday"] == 7).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * (df["weekday"] - 1) / 7)
    df["dow_cos"] = np.cos(2 * np.pi * (df["weekday"] - 1) / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)
    season_map = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
                  6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}
    df["season"] = df["month"].map(season_map)
    return df


# ---------- 3.2 conditional + 温度 合并 ----------
def merge_conditional(df: pd.DataFrame) -> pd.DataFrame:
    df = df.merge(holiday, on="date", how="left")
    df = df.merge(weather, on="date", how="left")
    df = df.merge(construction, on="date", how="left")
    df = df.merge(events, on="date", how="left")
    df = df.merge(temp_hourly[["date", "hour", "lt_mean", "fbt_mean", "fbt_min"]],
                  on=["date", "hour"], how="left")
    df = df.merge(temp_climo, on=["month", "hour"], how="left")
    df["lt_mean"] = df["lt_mean"].fillna(df["lt_mean_c"])
    df["fbt_mean"] = df["fbt_mean"].fillna(df["fbt_mean_c"])
    df["fbt_min"] = df["fbt_min"].fillna(df["fbt_min_c"])
    df = df.drop(columns=["lt_mean_c", "fbt_mean_c", "fbt_min_c"])
    for c in ["weather_source", "window_direction", "window_risk_level", "a8_direction", "a93_direction"]:
        if c in df.columns:
            df[c] = df[c].fillna("none").replace("", "none").astype(str)
    return df


print("特征函数就绪 ✔")

In [ ]:
# ---------- 3.3 构建连续小时面板（每个 site 无间断）----------
# TFT 需要每条序列有连续整数 time_idx。先把每个 site 的时间轴补齐到完整小时网格，
# 再 merge 观测值，缺测目标用「站点×小时×星期」中位数回填并打 is_imputed 标记。

site_meta = (
    traffic.groupby("site_id")[["road", "direction", "site_name", "bab_km", "longitude", "latitude"]]
    .first().reset_index()
)

GLOBAL_START = traffic["ts"].min().normalize()
GLOBAL_END = traffic["ts"].max().normalize() + pd.Timedelta(hours=23)
full_index = pd.date_range(GLOBAL_START, GLOBAL_END, freq="h")
print(f"完整小时轴: {full_index.min()} ~ {full_index.max()}  ({len(full_index):,} 步)")

# 每个 site × 完整小时轴
panel = pd.MultiIndex.from_product(
    [site_meta["site_id"], full_index], names=["site_id", "ts"]
).to_frame(index=False)

# 合并观测值
obs = traffic[["site_id", "ts", "kfz_h", "sv_h", "v_kfz", "tagestyp"]].drop_duplicates(["site_id", "ts"])
panel = panel.merge(obs, on=["site_id", "ts"], how="left")
panel = panel.merge(site_meta, on="site_id", how="left")

# 日历键
panel["date"] = panel["ts"].dt.normalize()
panel["hour"] = panel["ts"].dt.hour
panel["weekday"] = panel["ts"].dt.weekday + 1            # 1-7

# tagestyp 回填：先按 (site,date) 取已知值，再用 weekday 兜底
panel["tagestyp"] = panel.groupby(["site_id", "date"])["tagestyp"].transform(
    lambda s: s.ffill().bfill()
)
panel["tagestyp"] = panel["tagestyp"].fillna(np.where(panel["weekday"] == 7, "s", "w"))

# 缺测标记（目标缺失即为补值行）
panel["is_imputed"] = panel["kfz_h"].isna().astype(int)

# 目标/观测回填：站点×小时×星期 中位数 → 站点×小时 → 全局
def _fill_profile(df, col):
    for keys in (["site_id", "hour", "weekday"], ["site_id", "hour"]):
        med = df.groupby(keys)[col].transform("median")
        df[col] = df[col].fillna(med)
    df[col] = df[col].fillna(df[col].median())
    return df

for col in ["kfz_h", "sv_h", "v_kfz"]:
    panel = _fill_profile(panel, col)

# 日历 + conditional
panel = add_calendar(panel)
panel = merge_conditional(panel)

# conditional 数值列残余缺失补 0；类别列补 none
_num_fill = [
    "is_school_holiday_DE_BY", "is_school_holiday_AT_SB", "is_school_holiday_AT_TI",
    "is_public_holiday_DE_BY", "is_public_holiday_AT_SB", "is_public_holiday_AT_TI",
    "school_holiday_count", "public_holiday_count", "is_holiday_start", "is_holiday_end",
    "in_traffic_window", "w_precip", "w_snow", "w_lowvis", "w_tmin", "w_tmax", "w_ice",
    "lt_mean", "fbt_mean", "fbt_min",
    "has_a8_construction", "has_a93_construction", "a8_construction_count", "a93_construction_count",
    "has_2_plus_0", "two_plus_0_count", "max_closed_lanes", "sum_closed_lanes", "has_target_bbox_construction",
    "has_special_event", "active_event_count", "max_impact_level", "impact_score",
    "affects_a8_ost", "affects_a93_sued", "has_munich_event", "has_salzburg_event",
    "has_rosenheim_event", "has_kufstein_event", "has_confirmed_event", "has_estimated_event",
]
for c in _num_fill:
    if c in panel.columns:
        panel[c] = panel[c].fillna(0.0).astype(float)

# 连续整数 time_idx（按全局小时轴，保证所有 site 对齐且无间断）
panel["time_idx"] = ((panel["ts"] - GLOBAL_START) // pd.Timedelta(hours=1)).astype(int)
panel = panel.sort_values(["site_id", "time_idx"]).reset_index(drop=True)

# 类别列统一 str
for c in ["site_id", "road", "direction", "site_name", "tagestyp", "season",
          "weather_source", "window_direction", "window_risk_level", "a8_direction", "a93_direction"]:
    panel[c] = panel[c].astype(str)

print("连续面板:", panel.shape)
print("缺测回填比例: {:.1%}".format(panel["is_imputed"].mean()))
print("time_idx 范围:", panel["time_idx"].min(), "~", panel["time_idx"].max())
panel[["site_id", "ts", "time_idx", "kfz_h", "sv_h", "v_kfz", "tagestyp", "is_imputed"]].head()

In [ ]:
# ---------- 3.4 TFT 三类输入清单（对齐 model.md §7.1）----------
# A. Static — 不随时间变化
STATIC_CAT = ["site_id", "road", "direction", "site_name"]
STATIC_REAL = ["bab_km", "longitude", "latitude"]

# B. Time-varying KNOWN — 未来可知（日历 + 假期 + 天气气候态）
KNOWN_CAT = [
    "tagestyp", "season", "weather_source",
    "window_direction", "window_risk_level", "a8_direction", "a93_direction",
]
KNOWN_REAL = [
    # 日历
    "hour", "weekday", "month", "doy", "week_of_year",
    "is_weekend", "is_friday", "is_saturday", "is_sunday",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
    # 假期
    "is_school_holiday_DE_BY", "is_school_holiday_AT_SB", "is_school_holiday_AT_TI",
    "is_public_holiday_DE_BY", "is_public_holiday_AT_SB", "is_public_holiday_AT_TI",
    "school_holiday_count", "public_holiday_count",
    "is_holiday_start", "is_holiday_end", "in_traffic_window",
    # 天气气候态 + 温度
    "w_precip", "w_snow", "w_lowvis", "w_tmin", "w_tmax", "w_ice",
    "lt_mean", "fbt_mean", "fbt_min",
    # 施工
    "has_a8_construction", "has_a93_construction", "a8_construction_count", "a93_construction_count",
    "has_2_plus_0", "two_plus_0_count", "max_closed_lanes", "sum_closed_lanes", "has_target_bbox_construction",
    # 事件
    "has_special_event", "active_event_count", "max_impact_level", "impact_score",
    "affects_a8_ost", "affects_a93_sued",
    "has_munich_event", "has_salzburg_event", "has_rosenheim_event", "has_kufstein_event",
    "has_confirmed_event", "has_estimated_event",
]

# C. Time-varying OBSERVED — 仅历史可知（含目标）
UNKNOWN_REAL = [TARGET, "sv_h", "v_kfz"]

# 确保 KNOWN_REAL 全部为 float（TFT 连续变量要求）
for c in KNOWN_REAL:
    panel[c] = panel[c].astype(float)
for c in STATIC_REAL + UNKNOWN_REAL:
    panel[c] = panel[c].astype(float)

print(f"Static    : {len(STATIC_CAT)} cat + {len(STATIC_REAL)} real")
print(f"Known     : {len(KNOWN_CAT)} cat + {len(KNOWN_REAL)} real")
print(f"Observed  : {len(UNKNOWN_REAL)} real (含目标 {TARGET})")

## 4. 构建 `TimeSeriesDataSet` 与 DataLoader

- **时序切分**：训练集 = 截止 2024-12-31 的 `time_idx`；验证集 = 2025 全年（对齐 `model.ipynb`）
- **目标归一化**：按 `site_id` 分组（`GroupNormalizer` + softplus，保持正值）
- 验证集用 `from_dataset(..., predict=False, stop_randomization=True)` 继承训练集编码器/归一化参数

In [ ]:
# ---------- 4.1 时序切分 ----------
# 训练集 cutoff：TRAIN_END 对应的 time_idx
train_cutoff = int(panel.loc[panel["ts"] <= TRAIN_END, "time_idx"].max())
print(f"训练 time_idx 截止: {train_cutoff}  (≈ {TRAIN_END.date()})")

# 训练集：time_idx <= cutoff
train_panel = panel[panel["time_idx"] <= train_cutoff].copy()
# 验证集：需要包含编码器回看窗口，故从 cutoff - encoder 开始
val_panel = panel[panel["time_idx"] > train_cutoff - MAX_ENCODER_LENGTH].copy()

print(f"训练面板: {len(train_panel):>9,} 行")
print(f"验证面板: {len(val_panel):>9,} 行  (含 {MAX_ENCODER_LENGTH}h 编码器回看)")

# ---------- 4.2 训练集 TimeSeriesDataSet ----------
training = TimeSeriesDataSet(
    train_panel,
    time_idx="time_idx",
    target=TARGET,
    group_ids=["site_id"],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=STATIC_CAT,
    static_reals=STATIC_REAL,
    time_varying_known_categoricals=KNOWN_CAT,
    time_varying_known_reals=["time_idx"] + KNOWN_REAL,
    time_varying_unknown_reals=UNKNOWN_REAL,
    target_normalizer=GroupNormalizer(groups=["site_id"], transformation="softplus"),
    categorical_encoders={c: NaNLabelEncoder(add_nan=True) for c in STATIC_CAT + KNOWN_CAT},
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

# ---------- 4.3 验证集（继承训练集参数）----------
validation = TimeSeriesDataSet.from_dataset(
    training, val_panel, predict=False, stop_randomization=True,
)

train_loader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
val_loader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS)

print(f"\n训练样本: {len(training):>9,} | 验证样本: {len(validation):>9,}")
print(f"训练 batch: {len(train_loader)} | 验证 batch: {len(val_loader)}")

## 5. TFT 训练

- 损失：`QuantileLoss(QUANTILES)` → 直接输出 P10/P50/P90
- 回调：`EarlyStopping`（监控 val_loss）+ `ModelCheckpoint`（存最优）+ `LearningRateMonitor`
- 日志：`CSVLogger` → 训练后读 `metrics.csv` 画 loss 曲线
- 设备：`accelerator="auto"` 自动选 CUDA / Apple MPS / CPU

In [ ]:
# ---------- 5.1 构建 TFT 模型 ----------
tft = TemporalFusionTransformer.from_dataset(
    training,
    loss=QuantileLoss(quantiles=QUANTILES),
    logging_metrics=torch.nn.ModuleList([MAE(), RMSE()]),
    reduce_on_plateau_patience=REDUCE_ON_PLATEAU_PATIENCE,
    optimizer="adam",
    **TFT_PARAMS,
)
print(f"TFT 参数量: {tft.size()/1e3:.1f}k")

# ---------- 5.2 Trainer + 回调 ----------
early_stop = EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE, mode="min", verbose=True)
checkpoint = ModelCheckpoint(
    dirpath=str(TFT_DIR / "checkpoints"),
    filename="tft-{epoch:02d}-{val_loss:.3f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
lr_monitor = LearningRateMonitor(logging_interval="epoch")
csv_logger = CSVLogger(save_dir=str(LOG_DIR), name="tft")

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=ACCELERATOR,
    devices=1,
    gradient_clip_val=GRADIENT_CLIP_VAL,
    limit_train_batches=LIMIT_TRAIN_BATCHES,
    callbacks=[early_stop, checkpoint, lr_monitor],
    logger=csv_logger,
    enable_progress_bar=True,
    log_every_n_steps=20,
)

# ---------- 5.3 训练 ----------
trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)

best_path = checkpoint.best_model_path
print("\n✔ 训练完成")
print("  best checkpoint:", best_path)
print("  best val_loss  :", float(checkpoint.best_model_score) if checkpoint.best_model_score is not None else "n/a")

### 5.1 训练 / 验证 Loss 曲线

从 `CSVLogger` 的 `metrics.csv` 读取逐 epoch 的 train/val loss。

In [ ]:
for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    if _style in plt.style.available:
        plt.style.use(_style)
        break

metrics_path = Path(csv_logger.log_dir) / "metrics.csv"
m = pd.read_csv(metrics_path)

# 按 epoch 聚合 train/val loss
train_loss = m.dropna(subset=["train_loss"]).groupby("epoch")["train_loss"].mean() \
    if "train_loss" in m.columns else m.dropna(subset=["train_loss_epoch"]).groupby("epoch")["train_loss_epoch"].mean()
val_loss = m.dropna(subset=["val_loss"]).groupby("epoch")["val_loss"].mean()

fig, ax = plt.subplots(figsize=LOSS_FIGSIZE)
ax.plot(train_loss.index, train_loss.values, color="#2563eb", lw=1.8, marker="o", ms=4, label="train")
ax.plot(val_loss.index, val_loss.values, color="#dc2626", lw=1.8, marker="s", ms=4, label="validation")
best_ep = int(val_loss.idxmin())
ax.axvline(best_ep, color="#16a34a", ls="--", lw=1.2, alpha=0.8)
ax.scatter([best_ep], [val_loss.loc[best_ep]], color="#16a34a", zorder=5, label=f"best @ epoch {best_ep}")
ax.set_title("TFT · QuantileLoss（train / validation）", fontsize=13, fontweight="bold")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
ax.legend(frameon=True)
fig.tight_layout()
plt.show()

print(f"最优 epoch: {best_ep} | val_loss: {val_loss.loc[best_ep]:.4f}")

## 6. 验证集评估（2025 hold-out）

加载最优 checkpoint，对验证集多步预测，计算：
- P50 的 MAE / RMSE / MAPE
- P10–P90 区间覆盖率 PICP（目标 ≈ 80%）与平均区间宽度 MPIW
- 峰值小时 Recall（实测 top 10%）

In [ ]:
# ---------- 6.1 加载最优模型并预测 ----------
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_path) if best_path else tft

# 分位预测：output 形状 [N, pred_len, n_quantiles]
pred = best_tft.predict(
    val_loader, mode="quantiles",
    return_x=True, return_y=True, return_index=True,
)

# 兼容不同版本的返回结构
def _attr(p, name, pos):
    if hasattr(p, name):
        return getattr(p, name)
    return p[pos]

q_out = _attr(pred, "output", 0)
q_arr = q_out.cpu().numpy()                                  # [N, L, Q]
y_obj = _attr(pred, "y", 3)
y_t = y_obj[0] if isinstance(y_obj, (tuple, list)) else y_obj
y_arr = y_t.cpu().numpy()                                    # [N, L]
idx_df = _attr(pred, "index", 2).reset_index(drop=True)     # 每个样本的 (site_id, time_idx) 起点

print("分位预测:", q_arr.shape, "| 实测:", y_arr.shape, "| 索引:", idx_df.shape)

# 展平所有 (sample, step) 点
p10 = np.clip(q_arr[:, :, Q_IDX["p10"]].reshape(-1), KFZ_CLIP_MIN, None)
p50 = np.clip(q_arr[:, :, Q_IDX["p50"]].reshape(-1), KFZ_CLIP_MIN, None)
p90 = np.clip(q_arr[:, :, Q_IDX["p90"]].reshape(-1), KFZ_CLIP_MIN, None)
y_flat = y_arr.reshape(-1)


def mae(y, p): return float(np.mean(np.abs(y - p)))
def rmse(y, p): return float(np.sqrt(np.mean((y - p) ** 2)))
def mape(y, p, eps=MAPE_EPS):
    msk = y > eps
    return float(np.mean(np.abs((y[msk] - p[msk]) / y[msk])) * 100)


picp = float(np.mean((y_flat >= p10) & (y_flat <= p90)) * 100)
mpiw = float(np.mean(p90 - p10))
thr = np.quantile(y_flat, PEAK_QUANTILE)
true_peak = y_flat >= thr
pred_peak = p50 >= np.quantile(p50, PEAK_QUANTILE)
recall_peak = float(np.mean(pred_peak[true_peak])) if true_peak.any() else float("nan")

report = pd.DataFrame([
    {"目标": f"{TARGET} (TFT·P50)", "MAE": mae(y_flat, p50), "RMSE": rmse(y_flat, p50), "MAPE%": mape(y_flat, p50)},
]).set_index("目标").round(3)

print("\n验证集 (2025 hold-out) 评估")
display(report)
print(f"P10–P90 区间覆盖率 PICP : {picp:.1f}%  (目标≈{PICP_TARGET_PCT:.0f}%)")
print(f"P10–P90 平均区间宽度 MPIW: {mpiw:,.0f} 辆")
print(f"峰值小时 Recall (top{(1-PEAK_QUANTILE)*100:.0f}%)     : {recall_peak*100:.1f}%")

### 6.1 验证集示例：某站点一天 预测 vs 实测

取验证集中一个样本（24h 解码窗口），逐小时对比预测（P50 + P10–P90 区间）与真实流量。

In [ ]:
# 选一个解码窗口为整 24h、且实测非补值占比高的样本
_lengths = (~np.isnan(y_arr)).sum(axis=1)
_cand = np.where(_lengths >= MAX_PREDICTION_LENGTH)[0]
i = int(_cand[len(_cand) // 2]) if len(_cand) else 0

_site = idx_df.iloc[i]["site_id"]
_t0 = int(idx_df.iloc[i]["time_idx"])
_ts0 = GLOBAL_START + pd.Timedelta(hours=_t0)
_hours = np.arange(MAX_PREDICTION_LENGTH)

_p10 = np.clip(q_arr[i, :, Q_IDX["p10"]], KFZ_CLIP_MIN, None)
_p50 = np.clip(q_arr[i, :, Q_IDX["p50"]], KFZ_CLIP_MIN, None)
_p90 = np.clip(q_arr[i, :, Q_IDX["p90"]], KFZ_CLIP_MIN, None)
_y = y_arr[i, :]

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
ax.fill_between(_hours, _p10, _p90, color="#93c5fd", alpha=0.45, label="预测 P10–P90")
ax.plot(_hours, _p50, color="#1d4ed8", lw=2, marker="o", ms=4, label="预测 P50")
ax.plot(_hours, _y, color="#111827", lw=1.8, ls="--", marker="s", ms=4, label="实测 kfz_h")
ax.set_title(f"{_site} · 起 {_ts0:%Y-%m-%d %H:%M} 起 24h TFT 预测 vs 实测", fontsize=13, fontweight="bold")
ax.set_xlabel("解码步 (小时)")
ax.set_ylabel("kfz_h (辆/h)")
ax.set_xticks(range(0, MAX_PREDICTION_LENGTH, 2))
ax.legend(frameon=True)
fig.tight_layout()
plt.show()

_mae = float(np.mean(np.abs(_y - _p50)))
_picp1 = float(np.mean((_y >= _p10) & (_y <= _p90)) * 100)
print(f"该窗口 P50 MAE: {_mae:,.0f} 辆/h | P10–P90 覆盖率: {_picp1:.0f}% | {MAX_PREDICTION_LENGTH} 小时")

## 7. 可解释性（Variable Selection + Temporal Attention）

TFT 自带两类可解释输出（对齐 `SOLUTION.md` §6.3）：
- **变量选择权重**：encoder / decoder / static 各变量的重要性
- **时间注意力**：模型在编码器历史上关注的时间点

In [ ]:
# 用 raw 预测计算解释（取部分验证 batch 即可）
raw = best_tft.predict(val_loader, mode="raw", return_x=True)
raw_out = raw.output if hasattr(raw, "output") else raw[0]
raw_x = raw.x if hasattr(raw, "x") else raw[1]

interpretation = best_tft.interpret_output(raw_out, reduction="sum")
figs = best_tft.plot_interpretation(interpretation)
for _name, _fig in figs.items():
    _fig.suptitle(f"TFT 解释 · {_name}", fontsize=12, fontweight="bold")
    _fig.tight_layout()
plt.show()

# 文本版 Top 变量重要性
def _top_importance(weight_key, name_list, title, k=12):
    w = interpretation[weight_key].detach().cpu().numpy()
    s = pd.Series(w, index=name_list[: len(w)]).sort_values(ascending=False)
    print(f"\n{title} (Top {k}):")
    for n, v in s.head(k).items():
        print(f"  {n:32s} {v:8.2f}")

try:
    _top_importance("static_variables", best_tft.static_variables, "Static 变量重要性")
    _top_importance("encoder_variables", best_tft.encoder_variables, "Encoder 变量重要性")
    _top_importance("decoder_variables", best_tft.decoder_variables, "Decoder（已知未来）变量重要性")
except Exception as e:
    print("变量名映射不可用，已展示图形版解释:", type(e).__name__, e)

## 8. 模型保存与验证集预测落盘

- 保存最优 checkpoint 与 `TimeSeriesDataSet` 配置到 `models/tft/`
- 把验证集逐窗口分位预测写到 `processed/tft_val_forecast_2025.parquet`

> ⚠️ **关于 2026–2029 推理**：TFT 的 observed 输入（`kfz_h`/`sv_h`/`v_kfz`）远期不可得，需用历史画像填充编码器或做递归预测（误差累积）。按方案，**4 年主交付仍走 `model.ipynb` 的 CatBoost**；TFT 仅作历史回测对照。

In [ ]:
import shutil

# 1) 保存最优 checkpoint 副本
final_ckpt = TFT_DIR / "tft_best.ckpt"
if best_path and Path(best_path).exists():
    shutil.copy(best_path, final_ckpt)
else:
    trainer.save_checkpoint(str(final_ckpt))
print("✔ checkpoint ->", final_ckpt.relative_to(ROOT))

# 2) 保存 TimeSeriesDataSet 配置（推理时重建 dataset 用）
ds_params_path = TFT_DIR / "training_dataset.pkl"
training.save(str(ds_params_path))
print("✔ dataset 配置 ->", ds_params_path.relative_to(ROOT))

# 3) 验证集分位预测落盘（逐样本起点 × 解码步展开）
n, L, _ = q_arr.shape
rec = idx_df.loc[idx_df.index.repeat(L)].reset_index(drop=True)
rec["step"] = np.tile(np.arange(L), n)
rec["ts"] = GLOBAL_START + pd.to_timedelta(rec["time_idx"] + rec["step"], unit="h")
rec["kfz_h_p10"] = p10
rec["kfz_h_p50"] = p50
rec["kfz_h_p90"] = p90
rec["kfz_h_true"] = y_flat
rec["interval_width"] = rec["kfz_h_p90"] - rec["kfz_h_p10"]

val_fc_path = PROC_DIR / "tft_val_forecast_2025.parquet"
rec.to_parquet(val_fc_path, index=False)
print(f"✔ 验证集预测 {len(rec):,} 行 ->", val_fc_path.relative_to(ROOT))

print("\n已保存:")
print("  models/tft/tft_best.ckpt")
print("  models/tft/training_dataset.pkl")
print("  processed/tft_val_forecast_2025.parquet")
rec.head()